In [1]:
# Autoreload notebook extension
%load_ext autoreload
%autoreload 2

import sys

sys.path.append("..")
from src.ingestion import (
    load_documents_from_repo,
    chunk_documents_for_indexing,
    insert_chunks
)
from src.db.connection import fetch_all
from src.retrieval.retrievers import TextRetriever, VectorRetriever, HybridRetriever
from src.rag_pipeline import ask_rag


In [2]:
docs = load_documents_from_repo()

In [3]:
chunks = chunk_documents_for_indexing(docs)

In [4]:
chunks[0]

{'start': 0,
 'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phon

In [5]:
inserted = insert_chunks(chunks)
print(f"Inserted {inserted} chunks")

OperationalError: connection failed: connection to server at "127.0.0.1", port 5432 failed: Connection refused
	Is the server running on that host and accepting TCP/IP connections?

In [ ]:
fetch_all("""
            SELECT id, filename, start, content, embedding
            FROM rag.chunks
            ORDER BY id
            LIMIT 10
        """)

[(1,
  '01-agentic-rag/lessons/01-intro.md',
  0,
  '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most comm

In [10]:
query = "How to use LLM as a judge?"
search_text = TextRetriever().search(query=query, top_k=5)
search_vector = VectorRetriever().search(query=query, top_k=5)

In [11]:
search_text

[{'id': 174,
  'filename': '04-evaluation/lessons/11-evaluation-intro.md',
  'start': 1000,
  'content': "r, and also save the tool\ncalls. Then we can look at whether the answer is good and whether the\ntrajectory looks reasonable.\n\n## LLM as a judge\n\nFor RAG and agent evaluation, we compare the generated answer with the\noriginal answer. The generated answer won't use the same words as the\noriginal. It's a generative model, so the phrasing will be different\neven when the meaning is the same.\n\nThis is why we use another LLM to do the comparison. We show the judge\nthe question, the original answer, and the generated answer. Then we ask\nit to decide if they are semantically equivalent.\n\nThis approach is called LLM-as-a-judge. The evaluating LLM is the\njudge. It classifies each answer as good or bad and explains its\nreasoning. Asking the judge to explain why it made a decision generally\nproduces better classifications than asking for just the verdict.\n\nNext, we'll start 

In [12]:
search_vector

[{'id': 230,
  'filename': '05-monitoring/lessons/09-built-in-judge.md',
  'start': 0,
  'content': '# Built-in Judge\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=YLOLQyrMDuY&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous module we used an LLM as a judge for offline evaluation.\nOne LLM grades the output of another. We can run the same idea online.\n\nAfter each answer, we ask a judge whether it\'s relevant to the question.\nThat gives us an automatic quality signal on every response. We don\'t\nhave to wait for anyone to click thumbs up or down.\n\nThere\'s one real difference from the offline setup. Back then we had\nground truth: a reference answer to compare against. The judge could\ncheck our answer against that known-good one. Online we don\'t have it.\n\nThe judge now only sees the question and the answer. It has to make the\ncall on its own. That\'s a harder job, so in the instructions we describe\nmore carefully what a good answer looks like.\n\n#

In [19]:
search_hybrid = HybridRetriever().search(query=query, top_k=5)

In [20]:
search_hybrid

[{'id': 230,
  'filename': '05-monitoring/lessons/09-built-in-judge.md',
  'start': 0,
  'content': '# Built-in Judge\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=YLOLQyrMDuY&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous module we used an LLM as a judge for offline evaluation.\nOne LLM grades the output of another. We can run the same idea online.\n\nAfter each answer, we ask a judge whether it\'s relevant to the question.\nThat gives us an automatic quality signal on every response. We don\'t\nhave to wait for anyone to click thumbs up or down.\n\nThere\'s one real difference from the offline setup. Back then we had\nground truth: a reference answer to compare against. The judge could\ncheck our answer against that known-good one. Online we don\'t have it.\n\nThe judge now only sees the question and the answer. It has to make the\ncall on its own. That\'s a harder job, so in the instructions we describe\nmore carefully what a good answer looks like.\n\n#

In [24]:
ask_rag(
    query,
    top_k = 5,
    retriever_mode = "text",
)

{'answer': 'Use a second LLM to compare the **question**, the **original answer**, and the **generated answer**. Ask it whether the generated answer is **semantically equivalent** to the original, and have it return a **good/bad judgment plus reasoning**.\n\nKey idea:\n- The judge sees:\n  - the question\n  - the reference/original answer\n  - the model-generated answer\n- It decides if they mean the same thing, even if wording differs.\n- Asking the judge to **explain its decision** usually improves classification quality.\n\nThis is called **LLM-as-a-judge**.\n\n',
 'usage': {'model': 'gpt-5.4-mini',
  'input_tokens': 2306,
  'cached_tokens': 0,
  'output_tokens': 128,
  'reasoning_tokens': 0,
  'total_tokens': 2434},
 'cost': {'model': 'gpt-5.4-mini',
  'input_tokens': 2306,
  'output_tokens': 128,
  'input_price': Decimal('0.75'),
  'output_price': Decimal('4.50'),
  'input_cost': Decimal('0.00172950'),
  'output_cost': Decimal('0.00057600'),
  'total_cost': Decimal('0.00230550')},

In [13]:
ask_rag(
    query,
    top_k = 5,
    retriever_mode = "hybrid",
)

{'answer': 'Use an LLM-as-a-judge by having one LLM evaluate another model’s answer. For offline evaluation, the judge compares the question, the original answer, and the generated answer to decide whether they are semantically equivalent [04-evaluation/lessons/11-evaluation-intro.md:1000]. For production or online evaluation, the judge only sees the question and the generated answer, then classifies relevance as `RELEVANT`, `PARTLY_RELEVANT`, or `NON_RELEVANT` [05-monitoring/lessons/09-built-in-judge.md:0].\n\nThe lesson’s approach is:\n- define structured output for the verdict\n- give the judge clear instructions about what counts as relevant\n- pass in the question and answer\n- run the judge after each answer to get an automatic quality signal [05-monitoring/lessons/09-built-in-judge.md:0]\n\nA key point is that the judge should explain its reasoning, not just return a label, because that generally produces better classifications [04-evaluation/lessons/11-evaluation-intro.md:1000]